# Dataset and experimental design: manifest-level figures

Figures produced:

1. **Domain metadata overview** -- distribution of `road_type` and `time_of_day`
   across the full ZOD train split.
2. **Class prevalence by road type** -- Pedestrian / VulnerableVehicle / Vehicle
   presence rates across the five road types.
3. **Stream structure visualization** -- stripe plot of the two manifests we
   run experiments on: `cityday_road_type` (coarse, 5 blocks) and
   `cityday_curated` (fine-grained, 13 blocks).
4. **Curated manifest structure** -- per-block domain tags, frame counts, and
   the target shifts each block exposes.
5. **Bootstrap composition** -- side-by-side view of the bootstrap prefix for
   both manifests.
6. **Federated client partition** -- domain composition per client under
   domain-aligned partitioning (each client = one coherent deployment domain).

All figures are saved as PDF to `notes/figures/`.

In [ ]:
from __future__ import annotations

import json
import sys
from collections import Counter
from pathlib import Path
from typing import Any, Dict, List, Tuple

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

%matplotlib inline

_proj = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
sys.path.insert(0, str(_proj / "src"))

from stream_active_fl.analysis import runs as ah

ah.setup_notebook_style()
plt.rcParams.update({"text.usetex": False, "mathtext.fontset": "cm"})

PROJECT_ROOT = ah.find_project_root()
FIG_DIR = PROJECT_ROOT / "notes" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_DIR = ah.resolve_manifest_path(
    PROJECT_ROOT, "data/ZOD_frames_preprocessed/Frames_1600x480/manifest_cityday_road_type.json"
).parent

print("Manifest dir:", MANIFEST_DIR)
print("Figure dir:  ", FIG_DIR)

In [ ]:
def load(name: str) -> Dict[str, Any]:
    with open(MANIFEST_DIR / name) as f:
        return json.load(f)

# Two manifests in scope:
#   * cityday_road_type: coarse 5-block stream (one block per road_type),
#     5 000-frame city-day bootstrap, used for the main streaming + federated
#     runs where each block = one deployment domain.
#   * cityday_curated: fine-grained 13-block stream with a reduced 2 000-frame
#     bootstrap; exposes intra-city weather and time-of-day shifts before the
#     larger road-type shifts and is the target for the bootstrap + reservoir
#     adaptive-filter experiments.
MANIFESTS = {
    #"cityday_road_type": load("manifest_cityday_road_type.json"),
    "cityday_curated": load("manifest_cityday_curated_boot2000.json"),
}

REF = MANIFESTS["cityday_curated"]
ALL_FRAMES = REF["frames"]
BOOT_N = ah.get_bootstrap_size(REF)

ROAD_TYPE_ORDER = ["city", "arterial-urban", "highway", "arterial-rural", "smaller-rural"]
TOD_ORDER = ["day", "twilight", "night"]
TARGET_CLASSES = ["Vehicle", "Pedestrian", "VulnerableVehicle"]

# Colors / short-name maps come from stream_active_fl.analysis.runs.  The curated manifest
# uses compound block labels (e.g. city_day_clear) so we derive per-block
# colors from their road_type component at plot time.
DOMAIN_COLORS = ah.DOMAIN_COLORS
TOD_COLORS = ah.TOD_COLORS
ROAD_SHORT = ah.ROAD_SHORT
COND_SHORT = ah.WEATHER_SHORT


def _get_bootstrap_train_frames(mkey: str) -> List[Dict[str, Any]]:
    """Return the first N *train* frames from a manifest (skipping val)."""
    m = MANIFESTS[mkey]
    return ah.bootstrap_train_frames(m, ah.get_bootstrap_size(m))


print(f"Total frames: {len(ALL_FRAMES)}, bootstrap: {BOOT_N}")

## 1  Domain metadata overview

Distribution of `road_type` and `time_of_day` across the full ZOD train split
(all frames in the manifest, including bootstrap).

In [ ]:
train_frames = [f for f in ALL_FRAMES if f.get("split") == "train"]
n_train = len(train_frames)

rt_counts = Counter(f["road_type"] for f in train_frames)
tod_counts = Counter(f["time_of_day"] for f in train_frames)

fig, axes = plt.subplots(1, 2, figsize=(8, 2.8))

# Road type
ax = axes[0]
labels = [ROAD_SHORT.get(r, r) for r in ROAD_TYPE_ORDER]
vals = [rt_counts.get(r, 0) for r in ROAD_TYPE_ORDER]
colors = [DOMAIN_COLORS[r] for r in ROAD_TYPE_ORDER]
bars = ax.barh(labels, vals, color=colors, edgecolor="white", linewidth=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
            f"${v:,}$ ({v / n_train:.0%})", va="center", fontsize=7)
ax.set_xlabel("Number of frames")
ax.set_title(r"$\mathbf{Road\ type}$ distribution")
ax.invert_yaxis()
ax.set_xlim(0, max(vals) * 1.28)

# Time of day
ax = axes[1]
labels_tod = [t.capitalize() for t in TOD_ORDER]
vals_tod = [tod_counts.get(t, 0) for t in TOD_ORDER]
colors_tod = [TOD_COLORS[t] for t in TOD_ORDER]
bars = ax.barh(labels_tod, vals_tod, color=colors_tod, edgecolor="white", linewidth=0.5)
for bar, v in zip(bars, vals_tod):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
            f"${v:,}$ ({v / n_train:.0%})", va="center", fontsize=7)
ax.set_xlabel("Number of frames")
ax.set_title(r"$\mathbf{Time\ of\ day}$ distribution")
ax.invert_yaxis()
ax.set_xlim(0, max(vals_tod) * 1.28)

fig.suptitle(f"ZOD train split metadata ($n = {n_train:,}$ frames)", fontsize=11, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "domain_metadata_overview.pdf", bbox_inches="tight")
plt.show()

## 2  Target-class prevalence by road type

Fraction of frames containing each target class, broken down by `road_type`.
This motivates the domain-shift hypothesis: pedestrians are concentrated in
urban environments, while highway and rural areas see dramatically lower
pedestrian prevalence.

In [ ]:
rt_class_rates: Dict[str, Dict[str, float]] = {}
for rt in ROAD_TYPE_ORDER:
    rt_frames = [f for f in train_frames if f["road_type"] == rt]
    n = len(rt_frames)
    rt_class_rates[rt] = {}
    for cls in TARGET_CLASSES:
        count = sum(1 for f in rt_frames if cls in f.get("categories_present", []))
        rt_class_rates[rt][cls] = count / n if n > 0 else 0.0

CLASS_LABELS = {"Vehicle": "Vehicle", "Pedestrian": "Pedestrian", "VulnerableVehicle": "Vuln. Vehicle"}

fig, ax = plt.subplots(figsize=(7, 3.2))
x = np.arange(len(ROAD_TYPE_ORDER))
width = 0.25
class_colors = {"Vehicle": "#1f77b4", "Pedestrian": "#d62728", "VulnerableVehicle": "#ff7f0e"}

for i, cls in enumerate(TARGET_CLASSES):
    vals = [rt_class_rates[rt][cls] for rt in ROAD_TYPE_ORDER]
    bars = ax.bar(x + i * width, vals, width, label=CLASS_LABELS[cls],
                  color=class_colors[cls], edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals):
        if v > 0.05:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                    f"{v:.0%}", ha="center", va="bottom", fontsize=6)

ax.set_xticks(x + width)
ax.set_xticklabels([ROAD_SHORT.get(r, r) for r in ROAD_TYPE_ORDER])
ax.set_ylabel("Fraction of frames containing class")
ax.set_title("Target-class prevalence by road type")
ax.set_ylim(0, 1.12)
ax.legend(loc="upper center", ncol=3, framealpha=0.9, bbox_to_anchor=(0.5, 1.0))
ax.grid(axis="y", alpha=0.3)

fig.tight_layout()
fig.savefig(FIG_DIR / "class_prevalence_by_road_type.pdf", bbox_inches="tight")
plt.show()

## 3  Stream structure overview

High-level view of the stream: the bootstrap prefix followed by the five
road-type regions the stream walks through.  Consecutive curated blocks of
the same road type are merged into a single region for readability.  The
fine-grained 13-block ordering is expanded in section 4.

In [ ]:
_ROAD_TYPE_PREFIXES = {
    "city": "city",
    "arterial-urban": "arterial-urban",
    "highway": "highway",
    "arterial-rural": "arterial-rural",
    "smaller-rural": "smaller-rural",
}


def _block_road_type(bname: str) -> str:
    """Return the road_type component of a block name (plain or compound)."""
    for rt in _ROAD_TYPE_PREFIXES:
        if bname == rt or bname.startswith(rt + "_"):
            return rt
    return bname


def _block_color(bname: str) -> str:
    rt = _block_road_type(bname)
    return DOMAIN_COLORS.get(rt, "#888888")


def _block_label(bname: str) -> str:
    """Return a short display label for a block name."""
    rt = _block_road_type(bname)
    if bname == rt:
        return ROAD_SHORT.get(rt, rt)
    tail = bname[len(rt) + 1 :].replace("_", " ")
    return f"{ROAD_SHORT.get(rt, rt)}: {tail}"


o = REF["ordering"]
block_order = o["block_order"]
block_sizes = o["block_sizes"]
boot_n = ah.get_bootstrap_size(REF)
total_frames = boot_n + sum(block_sizes.values())

# Merge consecutive same-road-type blocks into regions for the overview.
regions: List[Tuple[str, int]] = []
for bname in block_order:
    rt = _block_road_type(bname)
    sz = block_sizes[bname]
    if regions and regions[-1][0] == rt:
        regions[-1] = (rt, regions[-1][1] + sz)
    else:
        regions.append((rt, sz))

fig, ax = plt.subplots(figsize=(12, 1.9))
bar_y = 0.0
bar_h = 0.7

ax.barh(bar_y, boot_n, left=0, height=bar_h,
        color="#cccccc", edgecolor="white", linewidth=0)
ax.text(boot_n / 2, bar_y, f"Bootstrap\n{boot_n:,}",
        ha="center", va="center", fontsize=9, color="#333")

pos = boot_n
for rt, sz in regions:
    ax.barh(bar_y, sz, left=pos, height=bar_h,
            color=DOMAIN_COLORS[rt], edgecolor="white", linewidth=0)
    frac = sz / total_frames
    if frac >= 0.08:
        ax.text(pos + sz / 2, bar_y, f"{ROAD_SHORT[rt]}\n{sz:,}",
                ha="center", va="center", fontsize=10,
                color="white", fontweight="bold")
    elif frac >= 0.04:
        ax.text(pos + sz / 2, bar_y, f"{ROAD_SHORT[rt]}\n{sz:,}",
                ha="center", va="center", fontsize=7,
                color="white", fontweight="bold")
    else:
        ax.text(pos + sz / 2, bar_y + bar_h / 2 + 0.04,
                f"{ROAD_SHORT[rt]} ({sz:,})",
                ha="left", va="bottom", fontsize=8,
                rotation=45, rotation_mode="anchor")
    pos += sz

ax.set_xlim(0, total_frames)
ax.set_ylim(-0.55, 0.55)
ax.set_yticks([])
ax.set_xlabel("Frame index")
ax.set_title(f"Stream structure -- bootstrap + 5 road-type regions "
             f"({total_frames:,} frames)")
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)

fig.tight_layout()
fig.savefig(FIG_DIR / "stream_structure.pdf", bbox_inches="tight")
plt.show()

## 4  Curated 13-block ordering

The `cityday_curated` manifest walks the stream through 13 blocks in a
deliberate order.  The first six blocks stay within the **city** road
type and expose condition and time-of-day shifts
(cloudy -> clear -> rain/wet -> snow -> twilight -> night).  The remaining
blocks introduce the larger road-type shifts
(arterial-urban -> highway -> arterial-rural -> smaller-rural), each split
by day vs. twilight/night where the counts support it.  The 2 000-frame
bootstrap is 100% city-day.

The figure below numbers the 13 blocks in stream order and lists each
block's label and frame count above the bar.

In [ ]:
CURATED = MANIFESTS["cityday_curated"]
curated_boot = ah.get_bootstrap_size(CURATED)
curated_train = [f for f in CURATED["frames"] if f.get("split") == "train"]
curated_stream = curated_train[curated_boot:]  # post-bootstrap stream

block_sizes = CURATED["ordering"]["block_sizes"]
block_order = CURATED["ordering"]["block_order"]

cursor = curated_boot
rows: List[Dict[str, Any]] = []
local = 0
for bname in block_order:
    sz = block_sizes[bname]
    block_frames = curated_stream[local : local + sz]
    rt = _block_road_type(bname)
    tod_c = Counter(f.get("time_of_day") for f in block_frames)
    cond_c = Counter(f.get("road_condition") for f in block_frames)
    rows.append({
        "block": bname,
        "road_type": rt,
        "dom_tod": tod_c.most_common(1)[0][0] if tod_c else "",
        "dom_cond": cond_c.most_common(1)[0][0] if cond_c else "",
        "frames": sz,
        "start_idx": cursor,
    })
    cursor += sz
    local += sz

print(f"{'block':<28} {'road_type':<16} {'dom. tod':<10} {'dom. cond':<10} "
      f"{'frames':>7} {'start':>8}")
print("-" * 82)
print(f"{'bootstrap':<28} {'city':<16} {'day':<10} {'normal':<10} "
      f"{curated_boot:>7,d} {0:>8,d}")
for r in rows:
    print(f"{r['block']:<28} {r['road_type']:<16} {r['dom_tod']:<10} "
          f"{r['dom_cond']:<10} {r['frames']:>7,d} {r['start_idx']:>8,d}")
print("-" * 82)
print(f"{'stream total':<28} {'':<16} {'':<10} {'':<10} "
      f"{sum(block_sizes.values()):>7,d}")

In [ ]:
curated_total = curated_boot + sum(block_sizes.values())

fig, ax = plt.subplots(figsize=(13, 4.8))
bar_y = 0.0
bar_h = 0.55

ax.barh(bar_y, curated_boot, left=0, height=bar_h,
        color="#cccccc", edgecolor="white", linewidth=0)
ax.text(curated_boot / 2, bar_y,
        f"Bootstrap\n(city-day, {curated_boot:,})",
        ha="center", va="center", fontsize=8, color="#333")

pos = curated_boot
for i, bname in enumerate(block_order, start=1):
    sz = block_sizes[bname]
    color = _block_color(bname)
    ax.barh(bar_y, sz, left=pos, height=bar_h,
            color=color, edgecolor="white", linewidth=0.6)
    ax.text(pos + sz / 2, bar_y, str(i),
            ha="center", va="center", fontsize=10, color="white",
            fontweight="bold")
    ax.text(pos + sz / 2, bar_y + bar_h / 2 + 0.05,
            f"{i}. {_block_label(bname)}  ({sz:,})",
            ha="left", va="bottom", fontsize=8,
            rotation=90, rotation_mode="anchor")
    pos += sz

ax.set_xlim(-500, curated_total + 500)
ax.set_ylim(-0.35, 2.4)
ax.set_yticks([])
ax.set_xlabel("Frame index")
ax.set_title(f"cityday_curated -- bootstrap + 13 ordered blocks "
             f"({curated_total:,} frames)")
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)

legend_handles = [mpatches.Patch(color="#cccccc", label="Bootstrap (city-day)")]
for rt in ROAD_TYPE_ORDER:
    if any(_block_road_type(b) == rt for b in block_order):
        legend_handles.append(
            mpatches.Patch(color=DOMAIN_COLORS[rt], label=ROAD_SHORT.get(rt, rt))
        )
ax.legend(handles=legend_handles, fontsize=8, framealpha=0.9,
          ncol=len(legend_handles), loc="lower center",
          bbox_to_anchor=(0.5, -0.22), frameon=False)

fig.tight_layout()
fig.savefig(FIG_DIR / "cityday_curated_structure.pdf", bbox_inches="tight")
plt.show()

## 5  Bootstrap composition

Both in-scope manifests bootstrap on city-day frames; they differ only in
prefix length (`cityday_road_type`: 5 000 frames, `cityday_curated`:
2 000 frames).  The panels below confirm the prefix is effectively a single
domain -- city roads in daylight with mostly normal road conditions -- so
any later accept-rate spike in the stream is unambiguously attributable to
a domain shift.

In [ ]:
#boot_rt = _get_bootstrap_train_frames("cityday_road_type")
boot_cur = _get_bootstrap_train_frames("cityday_curated")

fig, axes = plt.subplots(2, 2, figsize=(8, 4.5))
panels = [
    #(boot_rt, f"cityday_road_type (n = {len(boot_rt):,})"),
    (boot_cur, f"cityday_curated (n = {len(boot_cur):,})"),
]

for col, (bframes, blabel) in enumerate(panels):
    n = len(bframes)

    ax = axes[0, col]
    rt_c = Counter(f["road_type"] for f in bframes)
    labels = [ROAD_SHORT.get(r, r) for r in ROAD_TYPE_ORDER]
    vals = [rt_c.get(r, 0) for r in ROAD_TYPE_ORDER]
    colors = [DOMAIN_COLORS[r] for r in ROAD_TYPE_ORDER]
    bars = ax.barh(labels, vals, color=colors, edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
                    f"${v:,}$ ({v / n:.1%})", va="center", fontsize=6.5)
    ax.set_title(f"{blabel} -- road type")
    ax.invert_yaxis()
    ax.set_xlim(0, n * 1.2)

    ax = axes[1, col]
    tod_c = Counter(f["time_of_day"] for f in bframes)
    labels_tod = [t.capitalize() for t in TOD_ORDER]
    vals_tod = [tod_c.get(t, 0) for t in TOD_ORDER]
    colors_tod = [TOD_COLORS[t] for t in TOD_ORDER]
    bars = ax.barh(labels_tod, vals_tod, color=colors_tod, edgecolor="white", linewidth=0.5)
    for bar, v in zip(bars, vals_tod):
        if v > 0:
            ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
                    f"${v:,}$ ({v / n:.1%})", va="center", fontsize=6.5)
    ax.set_title(f"{blabel} -- time of day")
    ax.invert_yaxis()
    ax.set_xlim(0, n * 1.2)

fig.suptitle("Bootstrap composition", fontsize=11, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "bootstrap_composition.pdf", bbox_inches="tight")
plt.show()

## 6  Federated client partition

Federated runs use the `cityday_road_type` manifest with domain-aligned
partitioning: each of the 4 clients owns a coherent deployment environment
(urban taxi, suburban commuter, highway truck, rural delivery).  This yields
interpretable client-level heterogeneity and aligns each client's stream
with one (or at most two, for rural) manifest blocks.

In [ ]:
NUM_CLIENTS = 4

CLIENT_GROUPS = [
    ["city"],
    ["arterial-urban"],
    ["highway"],
    ["arterial-rural", "smaller-rural"],
]
CLIENT_LABELS = ["Client 0\n(City)", "Client 1\n(Suburban)", "Client 2\n(Highway)", "Client 3\n(Rural)"]

o = REF["ordering"]
block_order = o["block_order"]
block_sizes = o["block_sizes"]

block_offset: Dict[str, int] = {}
pos = 0
for b in block_order:
    block_offset[b] = pos
    pos += block_sizes[b]

client_ranges: List[Tuple[int, int]] = []
client_domain_frames: List[Dict[str, int]] = []
for group in CLIENT_GROUPS:
    starts = [block_offset[b] for b in group]
    ends = [block_offset[b] + block_sizes[b] for b in group]
    client_ranges.append((min(starts), max(ends)))
    client_domain_frames.append({b: block_sizes[b] for b in group})

fig, ax = plt.subplots(figsize=(9, 2.8))
row_height = 0.6
gap = 0.15

used_labels: set = set()
legend_handles: list = []

for ci, (group, (cs, ce)) in enumerate(zip(CLIENT_GROUPS, client_ranges)):
    y_bottom = ci * (row_height + gap)
    local_pos = 0
    for bname in group:
        sz = block_sizes[bname]
        c = _block_color(bname)
        ax.barh(y_bottom, sz, height=row_height, left=local_pos,
                color=c, edgecolor="white", linewidth=0.3)
        ax.text(local_pos + sz / 2, y_bottom + row_height / 2,
                f"{_block_label(bname)}\n({sz:,})",
                ha="center", va="center", fontsize=5.5, color="black")
        disp = _block_label(bname)
        if disp not in used_labels:
            legend_handles.append(mpatches.Patch(color=c, label=disp))
            used_labels.add(disp)
        local_pos += sz

y_positions = [i * (row_height + gap) + row_height / 2 for i in range(NUM_CLIENTS)]
ax.set_yticks(y_positions)
ax.set_yticklabels(CLIENT_LABELS, fontsize=7)
ax.set_xlabel("Frame count")
ax.set_title(f"Federated client partitioning ($K={NUM_CLIENTS}$, domain-aligned)")
ax.legend(handles=legend_handles, fontsize=6.5, framealpha=0.9,
          ncol=len(legend_handles),
          loc="lower center", bbox_to_anchor=(0.5, -0.32))

fig.tight_layout()
fig.savefig(FIG_DIR / "federated_client_partitions.pdf", bbox_inches="tight")
plt.show()

# Summary table
print(f"{'Client':<20} {'Frames':<8} {'Domains'}")
print("-" * 65)
for ci, (label, domains) in enumerate(zip(CLIENT_LABELS, client_domain_frames)):
    total = sum(domains.values())
    dom_str = ", ".join(f"{k}: {v:,}" for k, v in domains.items())
    print(f"Client {ci} {label.split(chr(10))[1]:<13} {total:<8,} {dom_str}")